In [4]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

In [5]:
RAW_PATH = (
    r"D:\adaptive_federated_gnn\data\raw\other datasets\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
)

PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Paths configured.")

Paths configured.


In [6]:
df = pd.read_csv(
    RAW_PATH,
    encoding="cp1252",
    low_memory=False
)

print("Original shape:", df.shape)

Original shape: (286467, 85)


In [7]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH 

In [8]:
print(df["Label"].value_counts())

Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [9]:
before_duplicates = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after_duplicates = len(df)

print("Rows before removing duplicates:", before_duplicates)
print("Rows after removing duplicates:", after_duplicates)
print("Duplicates removed:", before_duplicates - after_duplicates)

Rows before removing duplicates: 286467
Rows after removing duplicates: 286466
Duplicates removed: 1


In [10]:
df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinite values replaced.")

Infinite values replaced.


In [11]:
missing_before = df.isnull().sum().sum()

print("Missing values before cleaning:", missing_before)

Missing values before cleaning: 742


In [12]:
graph_columns = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label"
]

feature_columns = [
    column
    for column in df.columns
    if column not in graph_columns
]

for column in feature_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print("Feature columns converted to numeric.")

Feature columns converted to numeric.


In [13]:
before_missing = len(df)

df = df.dropna().reset_index(drop=True)

after_missing = len(df)

print("Rows before removing missing values:", before_missing)
print("Rows after removing missing values:", after_missing)
print("Rows removed:", before_missing - after_missing)

Rows before removing missing values: 286466
Rows after removing missing values: 286095
Rows removed: 371


In [14]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 0


In [15]:
df["Binary_Label"] = (
    df["Label"].str.strip().str.upper() != "BENIGN"
).astype(int)

print(df["Binary_Label"].value_counts())

Binary_Label
1    158804
0    127291
Name: count, dtype: int64


In [16]:
label_percentages = (
    df["Binary_Label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(label_percentages)

Binary_Label
1    55.51
0    44.49
Name: proportion, dtype: float64


In [17]:
metadata_columns = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label",
    "Binary_Label"
]

metadata_columns = [
    column
    for column in metadata_columns
    if column in df.columns
]

graph_metadata = df[metadata_columns].copy()

print("Graph metadata shape:", graph_metadata.shape)
graph_metadata.head()

Graph metadata shape: (286095, 9)


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Label,Binary_Label
0,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.12,35396,192.168.10.50,22,6,7/7/2017 1:00,BENIGN,0
1,192.168.10.16-192.168.10.50-60058-22-6,192.168.10.16,60058,192.168.10.50,22,6,7/7/2017 1:00,BENIGN,0
2,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.12,35396,192.168.10.50,22,6,7/7/2017 1:00,BENIGN,0
3,192.168.10.12-192.168.10.50-35398-22-6,192.168.10.12,35398,192.168.10.50,22,6,7/7/2017 1:00,BENIGN,0
4,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.50,22,192.168.10.12,35396,6,7/7/2017 1:00,BENIGN,0


In [18]:
graph_metadata.to_csv(
    f"{PROCESSED_DIR}/graph_metadata.csv",
    index=False
)

print("Graph metadata saved.")

Graph metadata saved.


In [19]:
columns_to_remove = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label"
]

feature_df = df.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("Feature data shape:", feature_df.shape)

Feature data shape: (286095, 78)


In [20]:
constant_columns = [
    column
    for column in feature_df.columns
    if feature_df[column].nunique() <= 1
]

print("Constant columns:")
print(constant_columns)

feature_df = feature_df.drop(
    columns=constant_columns
)

print("Shape after removing constant columns:", feature_df.shape)

Constant columns:
['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Shape after removing constant columns: (286095, 68)


In [21]:
X = feature_df.drop(
    columns=["Binary_Label"]
)

y = feature_df["Binary_Label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (286095, 67)
y shape: (286095,)


In [22]:
X = feature_df.drop(
    columns=["Binary_Label"]
)

y = feature_df["Binary_Label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (286095, 67)
y shape: (286095,)


In [23]:
print(X.dtypes.value_counts())

float64    36
int64      31
Name: count, dtype: int64


In [24]:
X = X.apply(
    pd.to_numeric,
    errors="coerce"
)

print("All features converted to numeric.")

All features converted to numeric.


In [25]:
before_invalid = len(X)

valid_rows = (
    X.notnull().all(axis=1)
    & np.isfinite(X).all(axis=1)
)

X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

after_invalid = len(X)

print("Rows before invalid-value removal:", before_invalid)
print("Rows after invalid-value removal:", after_invalid)
print("Invalid rows removed:", before_invalid - after_invalid)

Rows before invalid-value removal: 286095
Rows after invalid-value removal: 286095
Invalid rows removed: 0


In [26]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

print("Features standardized.")
print(X_scaled.head())

Features standardized.
   Flow Duration  Total Fwd Packets  Total Backward Packets  \
0      -0.187810           1.921572                1.429891   
1      -0.185393           1.921572                1.429891   
2      -0.245529          -0.126811               -0.089164   
3      -0.186117           1.921572                1.359237   
4      -0.245533          -0.126811               -0.053837   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                     1.301911                     0.083183   
1                     1.301911                     0.083183   
2                    -0.125195                    -0.053141   
3                     1.336196                     0.076910   
4                    -0.125195                    -0.053141   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0               1.142997              -0.427194                0.520358   
1               1.142997              -0.427194                0.5203

In [27]:
processed_df = X_scaled.copy()

processed_df["Binary_Label"] = y.values

print("Processed dataset shape:", processed_df.shape)

Processed dataset shape: (286095, 68)


In [28]:
print("Missing values:")
print(processed_df.isnull().sum().sum())

print("\nInfinite values:")
print(np.isinf(processed_df.select_dtypes(include=np.number)).sum().sum())

print("\nLabel distribution:")
print(processed_df["Binary_Label"].value_counts())

Missing values:
0

Infinite values:
0

Label distribution:
Binary_Label
1    158804
0    127291
Name: count, dtype: int64


In [29]:
processed_df.to_csv(
    f"{PROCESSED_DIR}/cleaned_data.csv",
    index=False
)

print("Cleaned dataset saved.")

Cleaned dataset saved.


In [30]:
summary = pd.DataFrame({
    "Metric": [
        "Original rows",
        "Original columns",
        "Final rows",
        "Final feature columns",
        "Duplicate rows removed",
        "Missing/invalid rows removed",
        "Constant columns removed",
        "Normal samples",
        "Attack samples"
    ],
    "Value": [
        before_duplicates,
        df.shape[1],
        processed_df.shape[0],
        processed_df.shape[1] - 1,
        before_duplicates - after_duplicates,
        before_invalid - after_invalid,
        len(constant_columns),
        int((y == 0).sum()),
        int((y == 1).sum())
    ]
})

summary

,Metric,Value
0,Original rows,286467
1,Original columns,86
2,Final rows,286095
3,Final feature columns,67
4,Duplicate rows removed,1
5,Missing/invalid rows removed,0
6,Constant columns removed,10
7,Normal samples,127291
8,Attack samples,158804


In [31]:
summary.to_csv(
    f"{PROCESSED_DIR}/preprocessing_summary.csv",
    index=False
)

print("Preprocessing summary saved.")

Preprocessing summary saved.


In [32]:
print("========== DAY 2 COMPLETE ==========")

print("\nCleaned data:")
print(processed_df.shape)

print("\nGraph metadata:")
print(graph_metadata.shape)

print("\nLabels:")
print(processed_df["Binary_Label"].value_counts())

print("\nMissing values:")
print(processed_df.isnull().sum().sum())

print("\nSaved files:")
print("1. data/processed/cleaned_data.csv")
print("2. data/processed/graph_metadata.csv")
print("3. data/processed/preprocessing_summary.csv")

========== DAY 2 COMPLETE ==========

Cleaned data:
(286095, 68)

Graph metadata:
(286095, 9)

Labels:
Binary_Label
1    158804
0    127291
Name: count, dtype: int64

Missing values:
0

Saved files:
1. data/processed/cleaned_data.csv
2. data/processed/graph_metadata.csv
3. data/processed/preprocessing_summary.csv
